In [ ]:
import pandas as pd
import numpy as np
import equiboots as eqb
import matplotlib.pyplot as plt
from equiboots.tables import metrics_table
from core.model_registry import best_per_algo, load_best_per_algo

In [ ]:
help(eqb)

## Read in Data and Model Object

In [ ]:
X = pd.read_parquet("../data/processed/X.parquet")
y = pd.read_parquet("../data/processed/y.parquet").squeeze()

In [ ]:
best_per_algo(metric="valid Average Precision")
champs = load_best_per_algo(metric="valid Average Precision")
model_catboost = champs["cat_outcome"]
model_catboost_no_sex = champs["cat_outcome_no_sex"]

In [ ]:
X_valid, y_valid = model_catboost_no_sex.get_valid_data(X, y)
X_test, y_test = model_catboost_no_sex.get_test_data(X, y)
y_test = y_test["outcome"]

In [ ]:
X_test_labeled = X_test.copy()
X_test_labeled["sex"] = X_test_labeled["sex"].map({1: "Male", 0: "Female"}).astype(str)

fairness_df = X_test_labeled[["sex"]].reset_index()

In [ ]:
X = pd.read_parquet("../data/processed/X.parquet")
y = pd.read_parquet("../data/processed/y.parquet")["outcome"].squeeze()
X_test, y_test = model_catboost.get_test_data(X, y)

## Bootstrap Estimates

Bootstrap estimates:m
- randomly sampling fairness_df, y_true, y_prob, and y_pred

In [ ]:
test_config = {
    "test_type": "bootstrap_test",
    "alpha": 0.05,
    "adjust_method": "bonferroni",
    "confidence_level": 0.95,
    "tail_type": "two_tailed",
    "metrics": [
        "Accuracy_diff",
        "Precision_diff",
        "Recall_diff",
        "F1_Score_diff",
        "Specificity_diff",
        "TP_Rate_diff",
        "FP_Rate_diff",
        "FN_Rate_diff",
        "TN_Rate_diff",
        "Prevalence_diff",
        "Predicted_Prevalence_diff",
        "ROC_AUC_diff",
        "Average_Precision_Score_diff",
        "Log_Loss_diff",
        "Brier_Score_diff",
        "Calibration_AUC_diff",
    ],
}

In [ ]:
y_test_arr = np.asarray(y_test)          # non-destructive; safe to re-run
int_list = np.linspace(0, len(y_test_arr), num=len(y_test_arr), dtype=int).tolist()


def run_audit(model, label):
    """Bootstrap fairness audit for one model. Returns (metrics, sig_tests)."""
    thr = model.threshold["average_precision"]
    y_prob = model.predict_proba(X_test)[:, 1]
    y_pred = (y_prob >= thr).astype(int)

    eq = eqb.EquiBoots(
        y_true=y_test_arr,
        y_pred=y_pred,
        y_prob=y_prob,
        fairness_df=fairness_df,
        fairness_vars=["sex"],
        seeds=int_list,
        reference_groups=["Male"],
        task="binary_classification",
        bootstrap_flag=True,
        num_bootstraps=5001,
        boot_sample_size=len(y_test_arr),
        group_min_size=1,
        balanced=False,
        stratify_by_outcome=True,
    )
    eq.set_fix_seeds(int_list)
    eq.grouper(groupings_vars=["sex"])

    boot_data = eq.slicer("sex")
    boot_metrics = eq.get_metrics(boot_data)
    diffs = eq.calculate_differences(boot_metrics, "sex")
    sig = eq.analyze_statistical_significance(
        metric_dict=boot_metrics,
        var_name="sex",
        test_config=test_config,
        differences=diffs,
    )
    print(f"{label}: threshold {thr:.3f}")
    return boot_metrics, diffs, sig

In [ ]:
boots_primary, diffs_primary, sig_primary = run_audit(model_catboost, "primary")
boots_ablated, diffs_ablated, sig_ablated = run_audit(model_catboost_no_sex, "ablated")

In [ ]:
def relabel(boots, prefix):
    """Prefix group names so two models can share one plot."""
    return [{f"{prefix}: {g}": m for g, m in rep.items()} for rep in boots]

combined = [
    {**a, **b}
    for a, b in zip(
        relabel(boots_primary, "1 With sex"),
        relabel(boots_ablated, "2 No sex"),
    )
]

In [ ]:
WITH_SEX = "#4C72B0"
NO_SEX = "#C44E52"
REF_GREY = "#888888"

# eq_plot_bootstrap_forest calls plt.show() internally, which detaches the
# figure under the inline backend. Intercept it so the figure survives and
# can be recolored.
captured = {}
real_show = plt.show
plt.show = lambda *a, **k: captured.setdefault("fig", plt.gcf())

try:
    eqb.eq_plot_bootstrap_forest(
        group_boot_metrics=combined,
        metric="ROC AUC",
        reference_group="1 With sex: Male",
        title="AUROC by sex, before and after removing sex",
        figsize=(8, 4.5),
        x_lim=(0.55, 1.02),
        sort_alphabetically=True,
    )
finally:
    plt.show = real_show

fig = captured["fig"]
ax = fig.axes[0]

labels = [t.get_text() for t in ax.get_yticklabels()]
print(labels)  # verify row order before trusting the color map

row_color = [WITH_SEX if "With sex" in lab else NO_SEX for lab in labels]

# error bars: three black line objects per row (span, left cap, right cap)
bars = [ln for ln in ax.get_lines() if ln.get_color() in ("k", "black")]
for i, ln in enumerate(bars):
    ln.set_color(row_color[i // 3])

# mean markers
for coll in ax.collections:
    coll.set_color(row_color)

# strip the numeric sort prefixes: "1 With sex: Male" -> "With sex: Male"
ax.set_yticklabels([lab.split(" ", 1)[1] for lab in labels])

# recolor the reference line so it is not mistaken for a group series
for ln in ax.get_lines():
    if ln.get_linestyle() == "--":
        ln.set_color(REF_GREY)

# the legend was built before the recolor, so update its handle to match
leg = ax.get_legend()
if leg is not None:
    handles = getattr(leg, "legend_handles", None) or leg.legendHandles
    for handle in handles:
        if handle.get_linestyle() == "--":
            handle.set_color(REF_GREY)

fig.savefig("../images/pdf_images/forest_auroc_combined.pdf",
            bbox_inches="tight")

In [ ]:
"""
Decomposition of the sex-based false positive rate disparity.

The observed FPR ratio factors exactly into three multiplicative components
(Chouldechova identity). Because the factors multiply, their shares are
additive on the log scale, so the stacked bars are drawn in log space. Drawing
the raw factors as a stacked bar would imply they add, which they do not.

Every number is computed from the confusion matrices rather than transcribed,
and the identity is asserted before anything is plotted.
"""

import math
from pathlib import Path

import matplotlib.pyplot as plt

# ---------------------------------------------------------------------------
# Test-set confusion matrices, primary model, threshold 0.080
# ---------------------------------------------------------------------------
MEN = dict(tp=10, fp=28, fn=6, tn=85)
WOMEN = dict(tp=4, fp=7, fn=3, tn=95)

OUT_PDF = "../images/pdf_images/figure_fpr_decomposition.pdf"
OUT_PNG = "../images/png_images/figure_fpr_decomposition.png"


def rates(cm):
    """Group rates, with the Chouldechova identity checked against observed FPR."""
    tp, fp, fn, tn = cm["tp"], cm["fp"], cm["fn"], cm["tn"]
    n = tp + fp + fn + tn
    p = (tp + fn) / n
    ppv = tp / (tp + fp)
    fnr = fn / (tp + fn)
    fpr = fp / (fp + tn)
    identity = (p / (1 - p)) * ((1 - ppv) / ppv) * (1 - fnr)
    assert abs(fpr - identity) < 1e-12, "identity does not reproduce FPR"
    return dict(n=n, p=p, ppv=ppv, fnr=fnr, fpr=fpr)


m, w = rates(MEN), rates(WOMEN)

f_prev = (m["p"] / (1 - m["p"])) / (w["p"] / (1 - w["p"]))
f_ppv = ((1 - m["ppv"]) / m["ppv"]) / ((1 - w["ppv"]) / w["ppv"])
f_fnr = (1 - m["fnr"]) / (1 - w["fnr"])
ratio = m["fpr"] / w["fpr"]

assert abs(f_prev * f_ppv * f_fnr - ratio) < 1e-9, "factors do not reproduce ratio"

logs = [math.log(f_prev), math.log(f_ppv), math.log(f_fnr)]
total_log = math.log(ratio)
shares = [x / total_log for x in logs]

print(f"FPR men   = {m['fpr']:.4f}   FPR women = {w['fpr']:.4f}")
print(f"ratio     = {ratio:.4f}")
print(f"  prevalence odds {f_prev:.4f}  share {shares[0]*100:.1f}%")
print(f"  PPV             {f_ppv:.4f}  share {shares[1]*100:.1f}%")
print(f"  FNR             {f_fnr:.4f}  share {shares[2]*100:.1f}%")

# ---------------------------------------------------------------------------
# Plot
# ---------------------------------------------------------------------------
COLORS = ["#4C72B0", "#DD8452", "#937860"]
LABELS = ["Prevalence odds", "Positive predictive value", "False negative rate"]

fig, (axA, axB) = plt.subplots(
    2, 1, figsize=(7.6, 5.0), gridspec_kw={"height_ratios": [1.15, 1]}
)

# Panel A: multiplicative chain on a log axis, so segment widths are log(factor)
left = 0.0
for lg, color, label, factor in zip(logs, COLORS, LABELS, [f_prev, f_ppv, f_fnr]):
    axA.barh(0, lg, left=left, height=0.5, color=color, edgecolor="white",
             linewidth=1.2, label=f"{label} ({factor:.2f}x)")
    if lg > 0.06:
        axA.text(left + lg / 2, 0, f"{factor:.2f}x", ha="center", va="center",
                 color="white", fontsize=10, fontweight="bold")
    left += lg

axA.set_xlim(-0.02, total_log * 1.04)
axA.set_ylim(-0.55, 0.75)
axA.set_yticks([])
tick_factors = [1.0, 1.5, 2.0, 2.5, 3.0, 3.61]
axA.set_xticks([math.log(f) for f in tick_factors])
axA.set_xticklabels([f"{f:.2f}x" for f in tick_factors], fontsize=9)
axA.set_xlabel("Cumulative multiplicative factor (log scale)", fontsize=9.5)
axA.set_title(
    f"Observed false positive rate ratio, men to women: "
    f"{m['fpr']:.3f} / {w['fpr']:.3f} = {ratio:.2f}x",
    fontsize=10.5, fontweight="bold", pad=10,
)
for side in ("top", "right", "left"):
    axA.spines[side].set_visible(False)
axA.legend(loc="upper center", bbox_to_anchor=(0.5, -0.42), ncol=3,
           fontsize=8.5, frameon=False)

# Panel B: share of the disparity, additive on the log scale
left = 0.0
for share, color in zip(shares, COLORS):
    axB.barh(0, share * 100, left=left, height=0.5, color=color,
             edgecolor="white", linewidth=1.2)
    if share > 0.05:
        axB.text(left + share * 100 / 2, 0, f"{share*100:.1f}%", ha="center",
                 va="center", color="white", fontsize=10, fontweight="bold")
    left += share * 100

axB.set_xlim(0, 100)
axB.set_ylim(-0.45, 0.78)
axB.set_yticks([])
axB.set_xticks([0, 20, 40, 60, 80, 100])
axB.set_xticklabels(["0%", "20%", "40%", "60%", "80%", "100%"], fontsize=9)
axB.set_xlabel("Share of the disparity (log scale)", fontsize=9.5)
axB.set_title("Attribution of the disparity", fontsize=10.5,
              fontweight="bold", pad=26)
for side in ("top", "right", "left"):
    axB.spines[side].set_visible(False)

axB.axvline(shares[0] * 100, color="#B00000", linewidth=1.4, linestyle="--",
            ymin=0.02, ymax=0.72)
axB.text(shares[0] * 100 / 2, 0.40,
         f"attributable to base rate ({shares[0]*100:.1f}%)",
         ha="center", va="bottom", fontsize=8.5, color="#333333")
axB.text(shares[0] * 100 + (100 - shares[0] * 100) / 2, 0.40,
         f"addressable ({(1-shares[0])*100:.1f}%)",
         ha="center", va="bottom", fontsize=8.5, color="#333333")

plt.tight_layout()
plt.subplots_adjust(hspace=1.35)

for path in (OUT_PDF, OUT_PNG):
    Path(path).parent.mkdir(parents=True, exist_ok=True)

fig.savefig(OUT_PDF, bbox_inches="tight")
fig.savefig(OUT_PNG, dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
import matplotlib.pyplot as plt

wanted_metrics = [
    "ROC_AUC_diff",
    "FP_Rate_diff",
    "Predicted_Prevalence_diff",
    "Recall_diff",
]


def relabel(diffs, suffix):
    """Rename the non-reference group so two models can share one panel.

    The numeric prefix forces the plotting order (primary before ablated) and
    is stripped from the legend afterwards.
    """
    return [{f"{g} ({suffix})": m for g, m in rep.items()} for rep in diffs]


combined_diffs = [
    {**a, **b}
    for a, b in zip(
        relabel(diffs_primary, "1 with sex"),
        relabel(diffs_ablated, "2 sex removed"),
    )
]

# eq_group_metrics_plot calls plt.show() internally, which detaches the figure
# under the inline backend. Intercept it so the legend can be tidied.
captured = {}
real_show = plt.show
plt.show = lambda *a, **k: captured.setdefault("fig", plt.gcf())

try:
    eqb.eq_group_metrics_plot(
        group_metrics=combined_diffs,
        metric_cols=wanted_metrics,
        name="sex",
        categories="all",
        figsize=(8, 8),
        plot_type="violinplot",
        color_by_group=True,
        cmap="tab10",
        max_cols=2,
        show_grid=False,
        strict_layout=True,
        disparities=True,
        show_pass_fail=False,
    )
finally:
    plt.show = real_show

fig = captured.get("fig", plt.gcf())

# strip the numeric sort prefixes from the legend
legends = list(fig.legends)
for ax in fig.axes:
    leg = ax.get_legend()
    if leg is not None:
        legends.append(leg)

for leg in legends:
    for txt in leg.get_texts():
        t = txt.get_text()
        txt.set_text(
            t.replace("1 with sex", "with sex").replace("2 sex removed", "sex removed")
        )

for path in ("../images/png_images/figureS4_disparity_violins.png",
             "../images/pdf_images/figureS4_disparity_violins.pdf"):
    Path(path).parent.mkdir(parents=True, exist_ok=True)

fig.savefig("../images/png_images/figureS4_disparity_violins.png",
            dpi=300, bbox_inches="tight")
fig.savefig("../images/pdf_images/figureS4_disparity_violins.pdf",
            bbox_inches="tight")

In [ ]:
def ci_table(sig, label):
    """Extract difference, interval and significance for each metric."""
    rows = []
    for metric, res in sig["Female"].items():
        lo, hi = res.confidence_interval or (None, None)
        rows.append({
            "metric": metric,
            f"{label}_diff": res.statistic,
            f"{label}_lo": lo,
            f"{label}_hi": hi,
            f"{label}_sig": res.is_significant,
        })
    return pd.DataFrame(rows).set_index("metric")


out = ci_table(sig_primary, "primary").join(ci_table(sig_ablated, "ablated"))
print(out.round(3).to_string())